In [ ]:

import os
import sys
import time
import json
import math
import tempfile
from pathlib import Path
from datetime import datetime

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import networkx as nx
from joblib import Parallel, delayed
from numba import njit

PROJECT_ROOT = Path('/content/drive/MyDrive/lock-in/high school lock-in/independent research/Influence Suppression and Consensus Cycling')
DATA_DIR = PROJECT_ROOT / 'Data'
CODE_DIR = PROJECT_ROOT / 'Code'

# Ensure directories exist
DATA_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = DATA_DIR / 'full_results_v2.csv'
CHECKPOINT_JSON = DATA_DIR / 'checkpoint_v2.json'
LOG_FILE = DATA_DIR / 'simulation_execution.log'

T = 500
WINDOW = 20
DELTA_T_VAR = 100
JSR_W = 100
BATCH_SIZE = 2000
N_JOBS = -1 -1
N_SEEDS = 30

# Regime Classification Thresholds
VAR_TEMP_CYCLE_THRESH = 1e-3
SPATIAL_VAR_CONSENSUS_THRESH = 1e-4

def log(msg):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    line = f"[{timestamp}] {msg}"
    with open(LOG_FILE, 'a') as f:
        f.write(line + '\n')
    print(line)

def get_param_hash(model, topology, N, P, gamma_stub, shape, K_frac, seed):
    """Creates a float-safe string representation of parameters to avoid JSON hash collisions."""
    return f"{model}_{topology}_{N}_{P:.2f}_{gamma_stub:.2f}_{shape}_{K_frac}_{seed}"

def load_checkpoint():
    if CHECKPOINT_JSON.exists():
        try:
            with open(CHECKPOINT_JSON, 'r') as f:
                return set(json.load(f))
        except Exception as e:
            log(f"Warning: Could not read checkpoint file ({e}). Starting fresh.")
            return set()
    return set()

def save_checkpoint_atomic(done_set):
    """Safely writes checkpoint using a temporary file and atomic replace."""
    with tempfile.NamedTemporaryFile('w', dir=DATA_DIR, delete=False) as tf:
        json.dump(list(done_set), tf)
        temp_name = tf.name
    os.replace(temp_name, CHECKPOINT_JSON)

def append_results_to_csv(results_list):
    valid = [r for r in results_list if r is not None]
    if not valid: return
    df = pd.DataFrame(valid)
    df.to_csv(OUTPUT_CSV, mode='a', header=not OUTPUT_CSV.exists(), index=False)

def get_file_size_mb(filepath):
    if filepath.exists():
        return os.path.getsize(filepath) / (1024 * 1024)
    return 0.0

def generate_graph(N, topology, seed):
    """Generates graphs with guaranteed connectivity and self-loops."""
    safe_seed = int(seed % (2**32))
    connected = False
    attempts = 0

    while not connected and attempts < 100:
        current_seed = safe_seed + attempts
        if topology == 'ER':
            G = nx.erdos_renyi_graph(N, 0.2, seed=current_seed)
        elif topology == 'BA':
            G = nx.barabasi_albert_graph(N, min(2, N - 1), seed=current_seed)
        elif topology == 'WS':
            k = min(4, N - 1)
            if k % 2 != 0: k = max(2, k - 1)
            G = nx.connected_watts_strogatz_graph(N, k, 0.1, seed=current_seed)
        elif topology == 'k-regular':
            k = min(4, N - 1)
            if (N * k) % 2 != 0: k = max(2, k - 1)
            try:
                G = nx.random_regular_graph(k, N, seed=current_seed)
            except nx.NetworkXError:
                G = nx.random_regular_graph(2, N, seed=current_seed) # Safe fallback
        else:
            raise ValueError(f"Unknown topology: {topology}")

        connected = nx.is_connected(G)
        attempts += 1

    adj = nx.to_numpy_array(G, dtype=np.float64)
    np.fill_diagonal(adj, 1.0) # Mandatory self-loops
    return adj

@njit(cache=True, fastmath=True)
def row_normalise(W):
    N = W.shape[0]
    for i in range(N):
        s = 0.0
        for j in range(N): s += W[i, j]
        if s > 1e-14:
            for j in range(N): W[i, j] /= s
        else:
            for j in range(N): W[i, j] = 0.0
            W[i, i] = 1.0
    return W

@njit(cache=True, fastmath=True)
def hajnal_coeff(W):
    N = W.shape[0]
    max_diff = 0.0
    for i in range(N):
        for j in range(i+1, N):
            diff = 0.0
            for k in range(N):
                diff += abs(W[i, k] - W[j, k])
            if diff > max_diff:
                max_diff = diff
    return 0.5 * max_diff

@njit(cache=True, fastmath=True)
def compute_penalty_factor(x, f_out, P, shape, symmetric, is_kin):
    N = len(x)
    s = 0.0
    for i in range(N): s += x[i]
    mean_x = s / N

    s2 = 0.0
    for i in range(N):
        diff = x[i] - mean_x
        s2 += diff * diff
    std_x = math.sqrt(s2 / N)

    if std_x < 1e-7: # Smooth boundary
        for i in range(N): f_out[i] = 1.0
        return

    for i in range(N):
        if is_kin[i]:
            f_out[i] = 1.0
            continue

        D = (x[i] - mean_x) / (std_x + 1e-8)
        val = abs(D) if symmetric else D

        if val > 0.0:
            if shape == 0: h = val
            elif shape == 1: h = 1.0 if val > 0.5 else 0.0
            else: h = math.tanh(val)
            f_out[i] = max(0.0, 1.0 - P * h)
        else:
            f_out[i] = 1.0

@njit(cache=True, fastmath=True)
def proposed_step(x, x0, W0_norm, gamma_stub, is_kin, W_out, x_out, f, H_out):
    N = len(x)
    for i in range(N):
        for j in range(N):
            if W0_norm[i, j] > 0.0:
                # Corrected: strict in-group mutual exemption
                if is_kin[i] and is_kin[j]:
                    W_out[i, j] = W0_norm[i, j]
                else:
                    W_out[i, j] = W0_norm[i, j] * f[j]
            else:
                W_out[i, j] = 0.0

    for i in range(N):
        s = 0.0
        for j in range(N): s += W_out[i, j]
        if s > 1e-14:
            for j in range(N): W_out[i, j] /= s
        else:
            for j in range(N): W_out[i, j] = 0.0
            W_out[i, i] = 1.0

    # Build affine operator for diagnostics and perform update
    for i in range(N):
        dot = 0.0
        for j in range(N):
            val = W_out[i, j]
            dot += val * x[j]
            # Construct the true affine mapping matrix
            H_out[i, j] = (1.0 - gamma_stub) * val
            if i == j:
                H_out[i, j] += gamma_stub
        x_out[i] = gamma_stub * x0[i] + (1.0 - gamma_stub) * dot

@njit(cache=True, fastmath=True)
def anti_expert_step(x, x0, W0_norm, gamma_stub, history_x, t, window, W_out, x_out, H_out):
    N = len(x)
    hist_idx = t % window
    for i in range(N): history_x[i, hist_idx] = x[i] # Correctly tracks state, not deviation
    effective = t + 1 if t < window else window

    for i in range(N):
        for j in range(N):
            if W0_norm[i, j] > 0.0:
                if effective < 2:
                    W_out[i, j] = W0_norm[i, j] # Burn-in safety
                else:
                    s = 0.0; s2 = 0.0
                    for k in range(effective):
                        val = history_x[j, k]
                        s += val
                        s2 += val * val
                    m = s / effective
                    var = max((s2 / effective) - (m * m), 1e-6)
                    W_out[i, j] = W0_norm[i, j] * (1.0 / var)
            else:
                W_out[i, j] = 0.0

    for i in range(N):
        s = 0.0
        for j in range(N): s += W_out[i, j]
        if s > 1e-14:
            for j in range(N): W_out[i, j] /= s
        else:
            for j in range(N): W_out[i, j] = 0.0
            W_out[i, i] = 1.0

    for i in range(N):
        dot = 0.0
        for j in range(N):
            val = W_out[i, j]
            dot += val * x[j]
            H_out[i, j] = (1.0 - gamma_stub) * val
            if i == j: H_out[i, j] += gamma_stub
        x_out[i] = gamma_stub * x0[i] + (1.0 - gamma_stub) * dot

@njit(cache=True, fastmath=True)
def inverse_reputation_step(x, x0, W0_norm, true_mu, gamma_stub, history_dev, t, window, W_out, x_out, H_out):
    N = len(x)
    hist_idx = t % window
    for i in range(N): history_dev[i, hist_idx] = abs(x[i] - true_mu) # Correctly tracks from true truth
    effective = t + 1 if t < window else window

    for i in range(N):
        for j in range(N):
            if W0_norm[i, j] > 0.0:
                s = 0.0
                for k in range(effective): s += history_dev[j, k]
                mad = s / effective
                W_out[i, j] = W0_norm[i, j] * (1.0 / (1.0 + mad))
            else:
                W_out[i, j] = 0.0

    for i in range(N):
        s = 0.0
        for j in range(N): s += W_out[i, j]
        if s > 1e-14:
            for j in range(N): W_out[i, j] /= s
        else:
            for j in range(N): W_out[i, j] = 0.0
            W_out[i, i] = 1.0

    for i in range(N):
        dot = 0.0
        for j in range(N):
            val = W_out[i, j]
            dot += val * x[j]
            H_out[i, j] = (1.0 - gamma_stub) * val
            if i == j: H_out[i, j] += gamma_stub
        x_out[i] = gamma_stub * x0[i] + (1.0 - gamma_stub) * dot

@njit(cache=True, fastmath=True)
def stubborn_mix_step(x, x0, W0_norm, gamma_vec, x_out, H_out):
    N = len(x)
    for i in range(N):
        dot = 0.0
        for j in range(N):
            val = W0_norm[i, j]
            dot += val * x[j]
            H_out[i, j] = (1.0 - gamma_vec[i]) * val
            if i == j: H_out[i, j] += gamma_vec[i]
        x_out[i] = gamma_vec[i] * x0[i] + (1.0 - gamma_vec[i]) * dot

def run_single_simulation(model, topology, N, P, gamma_stub, shape, K_frac, seed, warm_up=False):
    rng = np.random.default_rng(seed)
    adj = generate_graph(N, topology, seed)
    W0_norm = adj.copy()
    row_normalise(W0_norm)

    x0 = rng.uniform(-1.0, 1.0, size=N)
    true_mu = float(np.mean(x0))

    if K_frac == 'N': K_actual = N
    elif K_frac == 'N/2': K_actual = max(1, N // 2)
    elif K_frac == 'N/5': K_actual = max(1, N // 5)
    elif K_frac == 'N/10': K_actual = max(1, N // 10)
    else: K_actual = 0

    is_kin = np.zeros(N, dtype=np.bool_)
    if K_actual > 0 and model in ['proposed', 'symmetric', 'fixed_suppression']:
        kin_idx = rng.choice(N, size=K_actual, replace=False)
        is_kin[kin_idx] = True

    gamma_vec = np.full(N, gamma_stub, dtype=np.float64)
    if model == 'stubborn-mix':
        gamma_vec = np.zeros(N, dtype=np.float64)
        stubborn_idx = rng.choice(N, size=N // 2, replace=False)
        gamma_vec[stubborn_idx] = 1.0

    history_buffer = np.zeros((N, WINDOW), dtype=np.float64)
    x_buffer = np.zeros((DELTA_T_VAR, N), dtype=np.float64)

    x_curr = x0.copy()
    x_next = np.empty_like(x_curr)
    W = np.empty((N, N), dtype=np.float64)
    H_true = np.empty((N, N), dtype=np.float64)
    f = np.ones(N, dtype=np.float64)
    fixed_weights = np.ones(N, dtype=np.float64)

    if model == 'fixed_suppression':
        compute_penalty_factor(x0, fixed_weights, P, shape, False, is_kin)

    T_run = 2 if warm_up else T
    W_last100_arr = np.empty((JSR_W, N, N), dtype=np.float64)

    for t in range(T_run):
        # Buffer tracks beginning of step
        if not warm_up and t >= (T_run - DELTA_T_VAR):
            x_buffer[t % DELTA_T_VAR, :] = x_curr

        if model == 'proposed':
            compute_penalty_factor(x_curr, f, P, shape, False, is_kin)
            proposed_step(x_curr, x0, W0_norm, gamma_stub, is_kin, W, x_next, f, H_true)
        elif model == 'symmetric':
            compute_penalty_factor(x_curr, f, P, shape, True, is_kin)
            proposed_step(x_curr, x0, W0_norm, gamma_stub, is_kin, W, x_next, f, H_true)
        elif model == 'fixed_suppression':
            proposed_step(x_curr, x0, W0_norm, gamma_stub, is_kin, W, x_next, fixed_weights, H_true)
        elif model == 'anti-expert':
            anti_expert_step(x_curr, x0, W0_norm, gamma_stub, history_buffer, t, WINDOW, W, x_next, H_true)
        elif model == 'inverse-reputation':
            inverse_reputation_step(x_curr, x0, W0_norm, true_mu, gamma_stub, history_buffer, t, WINDOW, W, x_next, H_true)
        elif model == 'stubborn-mix':
            stubborn_mix_step(x_curr, x0, W0_norm, gamma_vec, x_next, H_true)

        x_curr, x_next = x_next, x_curr

        if not warm_up and t >= (T_run - JSR_W):
            W_last100_arr[t % JSR_W, :, :] = H_true # Store true affine mapping

    if warm_up: return None

    # Compute Verified Convergence Diagnostics
    Var_temp = float(np.mean(np.var(x_buffer, axis=0)))
    terminal_spatial_var = float(np.var(x_curr))
    consensus_error = float(abs(np.mean(x_curr) - true_mu))

    # Calculate Ergodicity on true Affine Product
    M = np.eye(N)
    for i in range(JSR_W):
        idx = (T_run - JSR_W + i) % JSR_W
        M = W_last100_arr[idx] @ M
    delta_product = hajnal_coeff(M)

    # Rigorous Classification
    if Var_temp >= VAR_TEMP_CYCLE_THRESH:
        regime = 'Regime 3 (Non-convergent Cycling)'
    elif terminal_spatial_var < SPATIAL_VAR_CONSENSUS_THRESH:
        regime = 'Regime 1 (Biased Convergence)'
    elif Var_temp < SPATIAL_VAR_CONSENSUS_THRESH and terminal_spatial_var >= SPATIAL_VAR_CONSENSUS_THRESH:
        regime = 'Regime 1B (Fixed Dissensus)'
    else:
        regime = 'Regime 2 (Slow Mixing)'

    return {
        'model': model,
        'topology': topology,
        'N': N,
        'P': P,
        'gamma_stub': gamma_stub,
        'penalty_shape': ['Linear', 'Step', 'Tanh'][shape] if model in ['proposed', 'symmetric', 'fixed_suppression'] else 'None',
        'K_fraction': K_frac if model in ['proposed', 'symmetric', 'fixed_suppression'] else 'None',
        'K_count': K_actual,
        'seed': seed,
        'delta_product': delta_product,
        'Var_temp': Var_temp,
        'terminal_spatial_var': terminal_spatial_var,
        'consensus_error': consensus_error,
        'regime': regime
    }

def build_pruned_grid():
    models = ['proposed', 'anti-expert', 'inverse-reputation', 'stubborn-mix', 'symmetric', 'fixed_suppression']
    topologies = ['ER', 'BA', 'WS', 'k-regular']
    N_vals = [20, 50, 100, 200]
    P_vals = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0]
    gamma_vals = [0.0, 0.1, 0.2, 0.5]
    shape_vals = [0, 1, 2]
    K_fractions = ['N', 'N/2', 'N/5', 'N/10']

    runs = []
    for model in models:
        for topo in topologies:
            for N in N_vals:
                if model in ['proposed', 'symmetric', 'fixed_suppression']:
                    for P in P_vals:
                        shapes = [0] if P == 0.0 else shape_vals
                        k_fracs = ['N'] if P == 0.0 else K_fractions
                        for K_frac in k_fracs:
                            if K_frac == 'N' and P > 0.0: continue
                            for gamma in gamma_vals:
                                for shape in shapes:
                                    for seed in range(N_SEEDS):
                                        runs.append((model, topo, N, P, gamma, shape, K_frac, seed))
                elif model == 'stubborn-mix':
                    for seed in range(N_SEEDS):
                        runs.append((model, topo, N, 0.0, 0.0, 0, 'None', seed))
                else:
                    for gamma in gamma_vals:
                        for seed in range(N_SEEDS):
                            runs.append((model, topo, N, 0.0, gamma, 0, 'None', seed))
    return runs

if __name__ == '__main__':
    log("--- Starting Simulation Engine V2.0 ---")
    log(f"Working Directory: {DATA_DIR}")

    all_runs = build_pruned_grid()
    log(f"Total parameter combinations (Pruned): {len(all_runs):,}")

    log("Performing JIT Compilation warm-up...")
    for m in ['proposed', 'anti-expert', 'inverse-reputation', 'stubborn-mix']:
        run_single_simulation(m, 'ER', 20, 0.1, 0.1, 0, 'N', 42, warm_up=True)
    log("Kernel warm-up complete.")

    completed_hashes = load_checkpoint()
    remaining = []
    for r in all_runs:
        param_hash = get_param_hash(*r)
        if param_hash not in completed_hashes:
            remaining.append(r)

    log(f"Resuming... Runs remaining: {len(remaining):,}")

    n_batches = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE
    start_time = time.time()

    for b_idx in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[b_idx:b_idx + BATCH_SIZE]
        batch_num = b_idx // BATCH_SIZE + 1
        log(f"Executing Batch {batch_num}/{n_batches} ({len(batch)} configurations)...")

        batch_start = time.time()
        results = Parallel(n_jobs=N_JOBS)(
            delayed(run_single_simulation)(*params) for params in batch
        )
        batch_duration = time.time() - batch_start

        # I/O sequence: Append CSV -> Atomic JSON write
        append_results_to_csv(results)

        for params in batch:
            completed_hashes.add(get_param_hash(*params))
        save_checkpoint_atomic(completed_hashes)

        file_mb = get_file_size_mb(OUTPUT_CSV)
        log(f"Batch {batch_num} saved. Duration: {batch_duration:.1f}s. CSV size: {file_mb:.2f} MB. Total complete: {len(completed_hashes):,}")

    total_time = (time.time() - start_time) / 3600
    log(f"Simulation Array Complete! Total execution time: {total_time:.2f} hours.")

Mounting Google Drive...
Mounted at /content/drive
[2026-09-13 14:50:29] --- Starting Simulation Engine V2.0 ---
[2026-09-13 14:50:29] Working Directory: /content/drive/MyDrive/independent research/Influence Suppression and Consensus Cycling/Data
[2026-09-13 14:50:29] Total parameter combinations (Pruned): 372,960
[2026-09-13 14:50:29] Performing JIT Compilation warm-up...
[2026-09-13 14:50:33] Kernel warm-up complete.
[2026-09-13 14:50:34] Resuming... Runs remaining: 372,960
[2026-09-13 14:50:34] Executing Batch 1/187 (2000 configurations)...
[2026-09-13 14:50:51] Batch 1 saved. Duration: 16.7s. CSV size: 0.29 MB. Total complete: 2,000
[2026-09-13 14:50:51] Executing Batch 2/187 (2000 configurations)...
[2026-09-13 14:51:03] Batch 2 saved. Duration: 12.2s. CSV size: 0.58 MB. Total complete: 4,000
[2026-09-13 14:51:03] Executing Batch 3/187 (2000 configurations)...
[2026-09-13 14:51:16] Batch 3 saved. Duration: 13.0s. CSV size: 0.87 MB. Total complete: 6,000
[2026-09-13 14:51:16] Execu

In [2]:
# =============================================================================
# STANDALONE DIAGNOSTIC & SUMMARY AUDITOR
# Run this cell independently at the bottom of your notebook.
# =============================================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    PROJECT_ROOT = Path('/content/drive/MyDrive/lock-in/high school lock-in/independent research/Influence Suppression and Consensus Cycling')
DATA_DIR = PROJECT_ROOT / 'Data'
CSV_PATH = DATA_DIR / 'full_results_v2.csv'

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not locate simulation CSV at: {CSV_PATH}")

print(f"Reading dataset from: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)

# Strip any residual whitespace or markdown escape slashes in column names
df.columns = [c.strip().replace(r'\_', '_') for c in df.columns]
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

print("\n" + "="*80)
print("             TALL POPPY SIMULATION V2: METRIC & CONSENSUS AUDIT")
print("="*80)

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
print("\n[1] EXECUTION INTEGRITY")
print(f"Total Rows Logged:          {len(df):,}")
print(f"Unique Models Tested:       {df['model'].unique().tolist()}")
print(f"Unique Topologies Tested:   {df['topology'].unique().tolist()}")
print(f"Unique Population Sizes N:  {sorted(df['N'].unique().tolist())}")
print(f"Missing Values (NaNs):      {df.isna().sum().to_dict()}")

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
print("\n[2] OVERALL REGIME BREAKDOWN (ALL MODELS)")
regime_counts = df['regime'].value_counts()
regime_pcts = df['regime'].value_counts(normalize=True) * 100
breakdown_df = pd.DataFrame({'Count': regime_counts, 'Percentage (%)': regime_pcts})
print(breakdown_df.to_string())

# -----------------------------------------------------------------------------
# Expectation: Zero penalty MUST strictly yield convergence (Regime 1)
# -----------------------------------------------------------------------------
print("\n[3] PROPOSED MODEL: ZERO-PENALTY INTEGRITY (P = 0.0)")
p0_df = df[(df['model'] == 'proposed') & (df['P'] == 0.0)]
if not p0_df.empty:
    p0_regimes = p0_df['regime'].value_counts()
    print(p0_regimes.to_string())
    print(f"P=0 Max Spatial Variance:  {p0_df['terminal_spatial_var'].max():.4e}")
    print(f"P=0 Max Temporal Variance: {p0_df['Var_temp'].max():.4e}")
else:
    print("No P=0.0 runs found for proposed model.")

# -----------------------------------------------------------------------------
# Shows the exact emergence of Regime 3 (Cycling) vs Regime 1B (Dissensus)
# -----------------------------------------------------------------------------
print("\n[4] PROPOSED MODEL: REGIME 3 (CYCLING) RATE OVER (P x gamma_stub)")
prop_df = df[df['model'] == 'proposed']
if not prop_df.empty:
    r3_matrix = prop_df.groupby(['gamma_stub', 'P'])['regime'].apply(
        lambda x: (x == 'Regime 3 (Non-convergent Cycling)').mean()
    ).unstack(fill_value=0.0)
    print("--- Probability of Regime 3 Cycling ---")
    print(r3_matrix.round(4).to_string())

    print("\n--- Probability of Regime 1B Fixed Dissensus ---")
    r1b_matrix = prop_df.groupby(['gamma_stub', 'P'])['regime'].apply(
        lambda x: (x == 'Regime 1B (Fixed Dissensus)').mean()
    ).unstack(fill_value=0.0)
    print(r1b_matrix.round(4).to_string())

# -----------------------------------------------------------------------------
# Tests whether larger clusters eliminate Regime 3 cycling
# -----------------------------------------------------------------------------
print("\n[5] KINSHIP IMMUNITY AUDIT (Proposed Model, gamma_stub = 0.5)")
kin_df = df[(df['model'] == 'proposed') & (df['gamma_stub'] == 0.5)]
if not kin_df.empty:
    kin_matrix = kin_df.groupby(['K_fraction', 'P'])['regime'].apply(
        lambda x: (x == 'Regime 3 (Non-convergent Cycling)').mean()
    ).unstack(fill_value=0.0)
    print(kin_matrix.round(4).to_string())

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
print("\n[6] CROSS-MODEL REGIME MATRIX (% OF RUNS)")
cross_model = pd.crosstab(df['model'], df['regime'], normalize='index') * 100
print(cross_model.round(2).to_string())

print("\n[7] MODEL CONVERGENCE & ACCURACY METRICS")
metric_summary = df.groupby('model').agg(
    Mean_Error=('consensus_error', 'mean'),
    Std_Error=('consensus_error', 'std'),
    Mean_Spatial_Var=('terminal_spatial_var', 'mean'),
    Mean_Temporal_Var=('Var_temp', 'mean'),
    Mean_Hajnal_Prod=('delta_product', 'mean')
)
print(metric_summary.to_string())

# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
print("\n[8] REGIME 3 CYCLING BY TOPOLOGY (PROPOSED MODEL, P >= 0.7)")
p_severe = df[(df['model'] == 'proposed') & (df['P'] >= 0.7)]
if not p_severe.empty:
    topo_r3 = p_severe.groupby('topology')['regime'].apply(
        lambda x: (x == 'Regime 3 (Non-convergent Cycling)').mean() * 100
    )
    print(topo_r3.round(2).to_string())

print("\n" + "="*80)
print("                          AUDIT COMPLETE")
print("="*80)

Reading dataset from: /content/drive/MyDrive/lock-in/high school lock-in/independent research/Influence Suppression and Consensus Cycling/Data/full_results_v2.csv

             TALL POPPY SIMULATION V2: METRIC & CONSENSUS AUDIT

[1] EXECUTION INTEGRITY
Total Rows Logged:          368,993
Unique Models Tested:       ['proposed', 'anti-expert', 'inverse-reputation', 'stubborn-mix', 'symmetric', '9135', 's)', 'fixed_suppression']
Unique Topologies Tested:   ['ER', 'BA', 'WS', 'k-regular', 'Regime 1B (Fixed Dissensus)', 'nan']
Unique Population Sizes N:  [20.0, 50.0, 100.0, 200.0, nan]
Missing Values (NaNs):      {'model': 0, 'topology': 0, 'N': 2, 'P': 2, 'gamma_stub': 2, 'penalty_shape': 0, 'K_fraction': 0, 'K_count': 2, 'seed': 2, 'delta_product': 2, 'Var_temp': 2, 'terminal_spatial_var': 2, 'consensus_error': 2, 'regime': 0}

[2] OVERALL REGIME BREAKDOWN (ALL MODELS)
                                    Count  Percentage (%)
regime                                                   
Regi